# Building a Decision Tree: Play Tennis Decision

## Overview

This notebook demonstrates how to build a **Decision Tree Classifier** for a practical problem with **categorical (text) features**.

**Problem:** Determine whether to play tennis based on weather conditions.

**Learning Objectives:**
1. Handle categorical (non-numeric) features in decision trees
2. Encode text categories to numeric values
3. Build a tree with mixed data types
4. Visualize and interpret the learned rules
5. Make predictions for new weather scenarios

**Dataset:**
- **Target:** Play Tennis (Yes/No)
- **Features:** Outlook, Temperature, Humidity, Wind (all categorical)
- **Samples:** 14 weather observations

## Step 1: Import Libraries and Create Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OrdinalEncoder

# Create the Play Tennis dataset (classic dataset from decision tree literature)
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play Tennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)
print('Dataset Overview:')
print(df)
print(f'\nDataset size: {len(df)} samples')
print(f'Target distribution: {df["Play Tennis"].value_counts().to_dict()}')

## Step 2: Handle Categorical Features

### Why Encoding?

Decision trees (and most ML algorithms) require **numeric input**. We have text categories like 'Sunny', 'Hot', etc.

### Encoding Approach: OrdinalEncoder

**OrdinalEncoder** converts each category to an integer:
- 'Sunny' → 2, 'Overcast' → 0, 'Rain' → 1 (alphabetical order)
- 'Hot' → 2, 'Mild' → 1, 'Cool' → 0 (alphabetical order)

**Note:** The numeric values don't mean anything ordinal (they're just labels). The decision tree will find the best splits regardless of the assigned numbers.

**Alternative:** One-hot encoding could also be used, but OrdinalEncoder is simpler for this example.

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Play Tennis', axis=1)  # Features
y = df['Play Tennis']               # Target

print('Features (before encoding):')
print(X.head())
print(f'\nFeature data types:\n{X.dtypes}')

# Encode categorical features to numeric values
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

print('\n' + '='*50)
print('Features (after OrdinalEncoder):')
print('\nFirst 3 rows (numeric):')  
print(X_encoded[:3])

print('\nEncoding mapping:')
for i, col in enumerate(X.columns):
    categories = encoder.categories_[i]
    print(f'{col}: {dict(enumerate(categories))}')

## Step 3: Train Decision Tree

### Key Hyperparameter: criterion

- `criterion='entropy'`: Uses information gain (based on entropy) to select splits
  - Common for classification tasks with categorical features
  - Better for understanding information-theoretic splits

- `criterion='gini'` (default): Uses Gini impurity
  - Slightly faster to compute
  - Usually produces similar trees

In [ ]:
# Create and train Decision Tree classifier
# Using entropy criterion for classic ID3-style decision tree
dt = DecisionTreeClassifier(
    criterion='entropy',   # Information gain based on entropy
    random_state=42        # For reproducibility
)

dt.fit(X_encoded, y)

print('Model Training Results:')
print(f'Tree depth: {dt.get_depth()}')
print(f'Number of leaves: {dt.get_n_leaves()}')
print(f'Number of nodes: {dt.tree_.node_count}')

## Step 4: Visualize the Learned Tree

### Reading the Tree

- **At each node:** Check if the condition is True → go left; False → go right
- **At leaf nodes:** The majority class is the prediction
- **Feature names:** Displayed in conditions (e.g., 'Outlook <= 1.5')
- **Colors:** Indicate the dominant class at that node
  - More saturated color = more confident (purer) prediction

In [ ]:
# Visualize the decision tree
plt.figure(figsize=(16, 10))
plot_tree(
    dt,
    feature_names=X.columns,      # Original feature names
    class_names=dt.classes_,      # Class labels (Yes/No)
    filled=True,                   # Color nodes by class dominance
    rounded=True,                  # Round node corners
    fontsize=10
)
plt.title('Play Tennis Decision Tree (Categorical Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5: Model Accuracy

Since we trained on the entire dataset (no train-test split), we evaluate on the same data.

**Note:** In production, always use a separate test set!

In [ ]:
# Make predictions
y_pred = dt.predict(X_encoded)

# Calculate accuracy
accuracy = (y_pred == y).sum() / len(y)

print(f'Training Accuracy: {accuracy:.2%}')
print(f'Correct Predictions: {(y_pred == y).sum()} out of {len(y)}')

# Show predictions vs actual
results_df = df.copy()
results_df['Predicted'] = y_pred
results_df['Correct'] = y_pred == y

misclassified = results_df[~results_df['Correct']]
if len(misclassified) > 0:
    print(f'\nMisclassified cases ({len(misclassified)}):')  
    print(misclassified[['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play Tennis', 'Predicted']])
else:
    print('\nPerfect predictions on training data!')

## Step 6: Make Predictions for New Scenarios

Let's predict whether to play tennis for some new weather conditions:

In [ ]:
# New weather scenarios to predict
new_scenarios = pd.DataFrame({
    'Outlook': ['Sunny', 'Overcast', 'Rain'],
    'Temperature': ['Cool', 'Mild', 'Cool'],
    'Humidity': ['Normal', 'High', 'Normal'],
    'Wind': ['Weak', 'Weak', 'Strong']
})

print('New Weather Scenarios:')
print(new_scenarios)

# Encode new scenarios
new_encoded = encoder.transform(new_scenarios)

# Make predictions
predictions = dt.predict(new_encoded)

# Get decision paths (which path through tree was taken)
decision_paths = dt.decision_path(new_encoded)

print('\nPredictions:')
for i, (idx, row) in enumerate(new_scenarios.iterrows()):
    print(f'{i+1}. {dict(row)} → Play Tennis: {predictions[i]}')

## Key Takeaways

1. **Categorical features need encoding** — decision trees require numeric input
2. **OrdinalEncoder** is simple for tree-based models (no ordinal assumption)
3. **Entropy-based splits** use information gain to reduce uncertainty
4. **Tree visualization** shows the exact rules the model learned
5. **Interpretability is a strength** of decision trees (unlike black-box models)

**Comparison with Other Notebooks:**
- Notebook 1 uses *numeric* features (Iris) — we used *categorical* features here
- Notebook 1 uses train-test split — we trained on all data for simplicity
- Both show how trees partition feature space through if-then rules

**Related Notebooks:**
- [Categorical Splits](6.categorical_split.ipynb) — understand how splits handle categories
- [Compute Entropy](4.compute_entropy.ipynb) — understand the entropy calculations
- [Decision Boundaries](5.decision_boundry.ipynb) — visualize decision regions